# FahMai OCR - TyphoonOCR + Qwen Pipeline

`renders/*.png|pdf` -> **submission.csv** (`artifact_id,pred_json`)

```
images -> [1 ROUTE] -> [2 FIND+PREFILL] -> [3a TYPHOON OCR] -> [3b QWEN EXTRACT] -> [4 NORMALIZE] -> [5 ASSEMBLE] -> submission.csv
```

**Changes from previous pipeline:**
- Removed: Surya OCR / llama.cpp / GGUF setup
- Added: **TyphoonOCR v1.5 2B** - image -> clean Markdown/text (runs locally on GPU)
- Updated: **Qwen2.5-3B-Instruct** - OCR text -> JSON field extraction (4-bit, Colab-friendly)
- ROUTE / PREFILL / NORMALIZE / ASSEMBLE logic is unchanged


## 0 · Install dependencies

In [ ]:
# Mount Google Drive (run this cell if files are on Drive)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install required packages
# TyphoonOCR 1.5 is based on Qwen3-VL, so keep transformers recent.
!pip install -q -U "transformers>=4.57.0" accelerate bitsandbytes pdf2image pillow
!apt-get install -q poppler-utils
print('Done')


In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/fahmai/super-ai-engineer-season-6-fah-mai-the-finale-ocr.zip"
extract_to = "/content/fahmai_data"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_to)

print(f"✅ Extracted to {extract_to}")

## 1 · Setup & Config

In [ ]:
import os, csv, glob, json, math, re, sys, html
from pathlib import Path

csv.field_size_limit(10**7)

# Paths - adjust to your Google Drive mount
ROOT = "/content/fahmai_data"
BASE = ROOT + "/fahmai_renders_with_json/fahmai_renders_with_json"
IDS  = ROOT + "/sample__submission.csv"
OUT  = ROOT + "/submission_surya_qwen.csv"  # kept for the same notebook output path/schema
PROV = BASE  + "/render_provenance.jsonl"  # dev-only; set "" for test mode

# OCR model: TyphoonOCR v1.5 2B is the best Colab fit in the TyphoonOCR family.
TYPHOON_OCR_MODEL_ID = "typhoon-ai/typhoon-ocr1.5-2b"
TYPHOON_MAX_IMAGE_SIZE = 1800
TYPHOON_MAX_NEW_TOKENS = 4096

# Qwen model for raw OCR text -> JSON. 3B 4-bit is safer than 7B when loaded with TyphoonOCR on Colab T4/L4.
QWEN_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
# Alternatives if VRAM is tight: "Qwen/Qwen2.5-1.5B-Instruct"
# Alternatives if VRAM is plenty: "Qwen/Qwen2.5-7B-Instruct"
QWEN_USE_4BIT = True
QWEN_MAX_NEW_TOKENS = 1024

print(f'ROOT        : {ROOT}')
print(f'OCR model   : {TYPHOON_OCR_MODEL_ID}')
print(f'Qwen model  : {QWEN_MODEL_ID}  (4-bit={QWEN_USE_4BIT})')


## 2 · Load Models

In [ ]:
# Load TyphoonOCR v1.5
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

print(f'Loading OCR model: {TYPHOON_OCR_MODEL_ID}...')

DEVICE_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

typhoon_processor = AutoProcessor.from_pretrained(
    TYPHOON_OCR_MODEL_ID,
    trust_remote_code=True,
)
typhoon_model = AutoModelForImageTextToText.from_pretrained(
    TYPHOON_OCR_MODEL_ID,
    trust_remote_code=True,
    torch_dtype=DEVICE_DTYPE,
    device_map="auto",
)
typhoon_model.eval()
print('TyphoonOCR ready')


In [ ]:
# Load Qwen2.5 for OCR text -> JSON extraction
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f'Loading JSON model: {QWEN_MODEL_ID}...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
) if QWEN_USE_4BIT else None

qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, trust_remote_code=True)
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    quantization_config=bnb_config,
)
qwen_model.eval()
print('Qwen ready')


## 3 · STEP 1 — ROUTE
Same as original: `artifact_id` → type + `renders/` subdir

In [ ]:
PREFIX_TO_TYPE = {
    "WC": ("WC", "warranty_form"), "VI": ("VI", "vendor_invoice"),
    "RC": ("RC", "receipt"),       "BS": ("BS", "bank_statement"),
    "BN": ("BN", "e7_banner"),     "T3": ("T3", "t3_doc"),
    "MEMO": ("MEMO", "t2_doc"), "VC": ("VC", "t2_doc"), "POL": ("POL", "t2_doc"),
    "EMAIL": ("EMAIL", "t2_doc"), "TRAIN": ("TRAIN", "t2_doc"), "AUD": ("AUD", "t2_doc"),
}

FIELDS = {
    "WC": ["claim_id", "business_event_date", "customer_id", "sku_id", "claim_reason", "claim_amount_thb"],
    "VI": ["payment_id", "vendor_id", "vendor_invoice_id", "invoice_period_start", "invoice_period_end", "paid_amount_thb", "business_event_date"],
    "RC": ["txn_id", "branch_code", "business_event_date", "basket_total_thb", "discount_total_thb", "net_total_thb", "payment_method"],
    "BS_HEADER": ["L0_account_id", "L0_bank", "L0_account_number", "L0_account_role", "L0_currency"],
    "BS_TXN": ["business_event_date", "transaction_type", "amount_thb", "balance_after_thb", "description", "account_id"],
    "MEMO": ["doc_id", "doc_kind", "template_name", "body_source", "issue_date"],
    "EMAIL": ["doc_id", "doc_kind", "template_name", "body_source", "issue_date"],
    "TRAIN": ["doc_id", "doc_kind", "template_name", "body_source", "issue_date"],
    "AUD": ["doc_id", "doc_kind", "template_name", "body_source", "issue_date"],
    "VC": ["contract_version_id", "vendor_id", "version_number", "effective_date", "contract_pdf_filename"],
    "POL": ["policy_version_id", "policy_class", "policy_variable", "scope_filter", "effective_date", "policy_doc_filename"],
    "T3": ["branch_code", "name_th", "name_en", "branch_type", "vendor_id", "category", "role", "payment_terms"],
    "BN": ["campaign_id", "description_th", "start_timestamp", "end_timestamp", "scope_filter"],
}

NOT_ON_PAGE = {
    "WC": {"claim_amount_thb"},
    "MEMO": {"template_name", "body_source"}, "EMAIL": {"template_name", "body_source"},
    "TRAIN": {"template_name", "body_source"}, "AUD": {"template_name", "body_source"},
    "VC": {"contract_pdf_filename"},
    "POL": {"policy_class", "policy_variable", "scope_filter", "policy_doc_filename"},
    "T3": {"name_en", "branch_type", "category", "role"},
    "BN": {"end_timestamp", "scope_filter"},
}

def route(artifact_id):
    for p in sorted(PREFIX_TO_TYPE, key=len, reverse=True):
        if artifact_id.startswith(p + "-"):
            return PREFIX_TO_TYPE[p]
    raise ValueError(f"unknown prefix: {artifact_id}")

print('✅ ROUTE defined')

## 4 · STEP 2 — FIND + PRE-FILL
Same as original: locate images + pre-fill ID fields from `artifact_id`

In [ ]:
def find_images(base, artifact_id):
    _, subdir = route(artifact_id)
    hits = glob.glob(os.path.join(base, "renders", subdir, "**", artifact_id + "*"), recursive=True)
    def order(p):
        b = os.path.basename(p)
        if "_header" in b: return (0, 0)
        m = re.search(r"_transactions_p(\d+)", b)
        return (1, int(m.group(1))) if m else (2, 0)
    return sorted(hits, key=order)

def prefill_ids(artifact_id):
    stype, _ = route(artifact_id)
    out = {}
    if stype == "VI":
        m = re.match(r"VI-(V-\d+)-(INV-\d+-\d+)$", artifact_id)
        if m: out["vendor_id"], out["vendor_invoice_id"] = m.group(1), f"{m.group(1)}-{m.group(2)}"
    elif stype == "WC":
        m = re.match(r"WC-(.+?)-(\d{6})-(\d+)$", artifact_id)
        if m:
            sku, yyyymm, tail = m.group(1), m.group(2), m.group(3)
            be, mm = int(yyyymm[:4]) + 543, yyyymm[4:6]
            out["sku_id"] = sku
            out["claim_id"] = f"WC-{be}-{mm}-{tail}"
            out["business_event_date"] = f"{tail[:2]}/{mm}/{be}"
            out["customer_id"] = f"CUST-L3-{tail[2:]}"
    elif stype == "RC":
        m = re.match(r"RC-(TXN-\d{6}-\d+)$", artifact_id)
        if m: out["txn_id"] = m.group(1)
    elif stype == "BN":
        out["campaign_id"] = artifact_id[len("BN-"):]
    return out

def period_yyyymm(artifact_id):
    m = re.search(r"-(\d{4})-(\d{2})$", artifact_id)
    return f"{int(m.group(1)) - 543}{m.group(2)}" if m else ""

PROVENANCE = {}
def load_provenance(path):
    if not path or not os.path.exists(path): return
    for line in open(path):
        d = json.loads(line)
        if "_transactions_p" in d.get("output_path", ""):
            PROVENANCE.setdefault(d["artifact_id"], d["source_row_ids"])
    print(f"[provenance] loaded txn ids for {len(PROVENANCE)} bank statements")

print('✅ FIND + PRE-FILL defined')

## 5 - STEP 3a - TYPHOON OCR
**New:** Image -> clean Markdown/text using TyphoonOCR v1.5 (replaces Surya OCR)


In [ ]:
from PIL import Image
from pdf2image import convert_from_path


TYPHOON_OCR_PROMPT = """Extract all text from the image.

Instructions:
- Only return the clean Markdown.
- Do not include any explanation or extra text.
- You must include all information on the page.
Formatting Rules:
- Tables: Render tables using <table>...</table> in clean HTML format.
- Equations: Render equations using LaTeX syntax with inline ($...$) and block ($$...$$).
- Images/Charts/Diagrams: Wrap any clearly defined visual areas (e.g. charts, diagrams, pictures) in:
<figure>
Describe the image's main elements (people, objects, text), note any contextual clues (place, event, culture), mention visible text and its meaning, provide deeper analysis when relevant (especially for financial charts, graphs, or documents), comment on style or architecture if relevant, then give a concise overall summary. Describe in Thai.
</figure>
- Page Numbers: Wrap page numbers in <page_number>...</page_number> (e.g., <page_number>14</page_number>).
- Checkboxes: Use ☐ for unchecked and ☑ for checked boxes.
"""


def load_image(image_path: str) -> list[Image.Image]:
    """Load image or PDF as a list of PIL Images."""
    if image_path.lower().endswith(".pdf"):
        return convert_from_path(image_path, dpi=200)
    return [Image.open(image_path).convert("RGB")]


def resize_if_needed(img: Image.Image, max_size: int = TYPHOON_MAX_IMAGE_SIZE) -> Image.Image:
    """TyphoonOCR v1.5 is trained around 1800 px image dimensions."""
    width, height = img.size
    longest = max(width, height)
    if longest <= max_size:
        return img
    scale = max_size / float(longest)
    new_size = (int(width * scale), int(height * scale))
    return img.resize(new_size, Image.Resampling.LANCZOS)


@torch.inference_mode()
def typhoon_ocr_image(pil_image: Image.Image) -> str:
    """Run TyphoonOCR on a single PIL Image and return Markdown/text."""
    img = resize_if_needed(pil_image)
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": TYPHOON_OCR_PROMPT},
            ],
        }
    ]

    inputs = typhoon_processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(typhoon_model.device)

    generated_ids = typhoon_model.generate(
        **inputs,
        max_new_tokens=TYPHOON_MAX_NEW_TOKENS,
        do_sample=False,
    )
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = typhoon_processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]
    return output_text.strip()


def typhoon_ocr_file(image_path: str) -> list[str]:
    """Run TyphoonOCR on image/PDF file and return one text block per page."""
    images = load_image(image_path)
    return [typhoon_ocr_image(img) for img in images]


print('TyphoonOCR helpers defined')


## 6 - STEP 3b - QWEN EXTRACT
**New:** OCR text -> structured JSON using Qwen2.5-Instruct (replaces Gemini JSON extraction)


In [ ]:
# ── Prompt templates (same intent as original _COMMON + _HINT) ───────────────
_COMMON = (
    "You are a data extraction assistant for Thai/English business documents.\n"
    "Below is OCR text extracted from a document page.\n"
    "Extract ONLY text actually present in the OCR output. "
    "If a field is not clearly present, return empty string \"\". "
    "Keep values exactly as found (Thai script, Buddhist years like 2567 as-is, "
    "keep comma/number formatting).\n"
    "Return ONLY a valid JSON object with exactly these keys: {keys}\n"
    "No markdown, no explanation — raw JSON only.\n"
)

_HINT = {
    "WC": "Warranty claim form. Only extract claim_reason (small vocab e.g. defect/damage). Others come from the id.",
    "VI": "A4 vendor invoice. payment_id = the BANK PAYMENT reference printed (starts 'BT-...'), NOT the invoice number. period start/end from line-items header; paid_amount_thb = Grand Total/เงินรวม; business_event_date = ลงวันที่.",
    "RC": "Thermal receipt. Extract branch_code, business_event_date, basket_total_thb, discount_total_thb (0.00 if none), net_total_thb, payment_method.",
    "BS_HEADER": "Bank statement HEADER page. Extract L0_account_id, L0_bank, L0_account_number, L0_account_role, L0_currency.",
    "BS_TXN": (
        "Bank statement TRANSACTION page. Return {\"rows\":[...]} — one JSON object per visible transaction line, "
        "top to bottom, skipping header/summary/total lines. "
        "Keys per row: business_event_date, transaction_type, amount_thb, balance_after_thb, description, account_id."
    ),
    "MEMO": "Internal memo/email/training/audit document. Extract doc_id, doc_kind, template_name, body_source, issue_date.",
    "EMAIL": "Email document. Extract doc_id, doc_kind, template_name, body_source, issue_date.",
    "TRAIN": "Training document. Extract doc_id, doc_kind, template_name, body_source, issue_date.",
    "AUD": "Audit document. Extract doc_id, doc_kind, template_name, body_source, issue_date.",
    "VC": "Vendor contract. Extract contract_version_id, vendor_id, version_number, effective_date, contract_pdf_filename.",
    "POL": "Policy document. Extract policy_version_id, policy_class, policy_variable, scope_filter, effective_date, policy_doc_filename.",
    "T3": "Branch/vendor T3 doc. Extract branch_code, name_th, name_en, branch_type, vendor_id, category, role, payment_terms.",
    "BN": "Banner/campaign doc. Extract campaign_id, description_th, start_timestamp, end_timestamp, scope_filter.",
}


def _build_prompt(ocr_text: str, type_key: str, keys: list[str], rows_mode: bool = False) -> str:
    """Build the Qwen extraction prompt from OCR text."""
    system = _COMMON.format(keys=", ".join(keys))
    hint = _HINT.get(type_key, "")
    if rows_mode:
        system = (
            "You are a data extraction assistant for Thai/English business documents.\n"
            "Below is OCR text from a bank statement transaction page.\n"
            f"{hint}\n"
            "Return ONLY a valid JSON object with key 'rows' containing a list of transaction objects.\n"
            "No markdown, no explanation — raw JSON only.\n"
        )
        hint = ""
    return system + (f"\nHint: {hint}\n" if hint else "") + f"\n--- OCR TEXT ---\n{ocr_text}\n--- END ---\n"


@torch.inference_mode()
def qwen_extract(ocr_text: str, type_key: str, keys: list[str], rows_mode: bool = False) -> dict:
    """
    Use Qwen2.5-Instruct to extract structured fields from TyphoonOCR text.
    Maps to: original extract() but receives text instead of image path.
    """
    prompt = _build_prompt(ocr_text, type_key, keys, rows_mode)

    messages = [
        {"role": "system", "content": "You are a precise document data extraction assistant. Always return valid JSON only."},
        {"role": "user",   "content": prompt},
    ]

    text = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = qwen_tokenizer([text], return_tensors="pt").to(qwen_model.device)

    output_ids = qwen_model.generate(
        **inputs,
        max_new_tokens=QWEN_MAX_NEW_TOKENS,
        do_sample=False,          # greedy — deterministic like temperature=0
        pad_token_id=qwen_tokenizer.eos_token_id,
    )
    # Decode only the newly generated tokens
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    raw = qwen_tokenizer.decode(new_ids, skip_special_tokens=True).strip()

    # Strip markdown fences if present
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw).strip()

    try:
        parsed = json.loads(raw)
        if rows_mode and isinstance(parsed, list):
            return {"rows": parsed}
        return parsed
    except json.JSONDecodeError:
        # Try to extract first JSON object/array
        m = re.search(r"(\{.*\}|\[.*\])", raw, re.DOTALL)
        if m:
            try:
                parsed = json.loads(m.group(1))
                if rows_mode and isinstance(parsed, list):
                    return {"rows": parsed}
                return parsed
            except Exception:
                pass
        print(f"[qwen-parse-error] type={type_key} raw={raw[:200]}", file=sys.stderr)
        return {"rows": []} if rows_mode else {k: "" for k in keys}


print('✅ Qwen extraction helpers defined')

## 7 - STEP 3 - EXTRACT (TyphoonOCR -> Qwen)
**Unified extract function** replacing original Gemini-based `extract()`


In [ ]:
def extract(image_path: str, type_key: str, keys: list[str], rows_mode: bool = False) -> dict:
    """
    Drop-in replacement for original extract().
    1. TyphoonOCR : image_path -> page text(s)
    2. Qwen LLM   : text + prompt -> JSON dict

    For multi-page PDFs, pages are concatenated with a separator.
    BS_TXN pages are usually single-page images.
    """
    try:
        page_texts = typhoon_ocr_file(image_path)
    except Exception as e:
        print(f"[typhoon-error] {os.path.basename(image_path)}: {e}", file=sys.stderr)
        return {"rows": []} if rows_mode else {k: "" for k in keys}

    full_text = "\n\n--- PAGE BREAK ---\n\n".join(page_texts).strip()

    if not full_text:
        print(f"[typhoon-empty] {os.path.basename(image_path)}", file=sys.stderr)
        return {"rows": []} if rows_mode else {k: "" for k in keys}

    try:
        return qwen_extract(full_text, type_key, keys, rows_mode)
    except Exception as e:
        print(f"[qwen-error] {os.path.basename(image_path)}: {e}", file=sys.stderr)
        return {"rows": []} if rows_mode else {k: "" for k in keys}


print('extract() (TyphoonOCR + Qwen) defined')


## 8 · STEP 4 — NORMALIZE / VALIDATE
Identical to original pipeline

In [ ]:
_THAI = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
_NUM  = re.compile(r"^-?\d{1,3}(,\d{3})*(\.\d{1,2})?$")

AMOUNT_FIELDS = {
    "paid_amount_thb", "basket_total_thb", "discount_total_thb",
    "net_total_thb", "claim_amount_thb", "amount_thb", "balance_after_thb"
}
PAYMENT_METHOD = {
    "cash": "เงินสด", "credit card": "บัตรเครดิต",
    "promptpay": "พร้อมเพย์", "qr": "QR"
}

def clean_amount(s):
    if not s: return ""
    s = s.translate(_THAI).strip()
    s = re.sub(r"[฿\s]|บาท|THB", "", s)
    s = re.sub(r"^[^\d-]+", "", s)
    if _NUM.match(s): return s
    digits = re.sub(r"[^\d.-]", "", s)
    try:    return f"{float(digits):,.2f}"
    except ValueError: return ""

def _to_float(s):
    try:    return float(re.sub(r"[^\d.-]", "", (s or "").translate(_THAI)))
    except ValueError: return None

def reconcile_balances(rows):
    prev = None
    for r in rows:
        amt, bal = _to_float(r.get("amount_thb")), _to_float(r.get("balance_after_thb"))
        debit = str(r.get("transaction_type", "")).lower() in {"withdrawal", "debit", "ถอน", "payment"}
        if prev is not None and amt is not None and bal is not None:
            if abs((prev + (-amt if debit else amt)) - bal) > 0.01:
                r["amount_thb"] = f"{abs(bal - prev):,.2f}"
                r["_corrected"] = True
        prev = bal if bal is not None else prev
    return rows

def norm_payment(s):
    return PAYMENT_METHOD.get((s or "").strip().lower(), (s or "").strip())

print('✅ NORMALIZE / VALIDATE defined')

## 9 · STEP 5 — ASSEMBLE
Identical to original pipeline

In [ ]:
def assemble(artifact_id, base):
    stype, _ = route(artifact_id)
    imgs = find_images(base, artifact_id)

    if stype == "BS":
        pred = {}
        header_imgs = [im for im in imgs if "_header" in im]
        txn_imgs    = [im for im in imgs if "_header" not in im]

        for img in header_imgs:
            hdr = extract(img, "BS_HEADER", FIELDS["BS_HEADER"])
            pred.update({k: hdr.get(k, "") for k in FIELDS["BS_HEADER"]})

        txns  = PROVENANCE.get(artifact_id)
        P     = len(txn_imgs)
        chunk = math.ceil(len(txns) / P) if (txns and P) else 0

        for p, img in enumerate(txn_imgs):
            rows = reconcile_balances(
                extract(img, "BS_TXN", FIELDS["BS_TXN"], rows_mode=True).get("rows", [])
            )
            if txns:
                ids = txns[p * chunk:(p + 1) * chunk]
            else:
                period = period_yyyymm(artifact_id)
                ids = [f"BT-{period}-row{i + 1}" for i in range(len(rows))]

            for i, txn in enumerate(ids):
                r   = rows[i] if i < len(rows) else {}
                key = f"L{p + 1}_{txn}_"
                for k in FIELDS["BS_TXN"]:
                    v = r.get(k, "")
                    pred[key + k] = clean_amount(v) if k in AMOUNT_FIELDS else v

        return {"artifact_id": artifact_id, "pred_json": json.dumps(pred, ensure_ascii=False)}

    # ── All other document types ─────────────────────────────────────────────
    keys     = FIELDS[stype]
    raw      = extract(imgs[0], stype, keys) if imgs else {}
    prefilled = prefill_ids(artifact_id)
    absent   = NOT_ON_PAGE.get(stype, set())
    pred     = {}

    for k in keys:
        if k in prefilled:
            pred[k] = prefilled[k]
        elif k in absent:
            pred[k] = ""
        else:
            v = raw.get(k, "")
            if k in AMOUNT_FIELDS:      v = clean_amount(v)
            elif k == "payment_method": v = norm_payment(v)
            pred[k] = v

    return {"artifact_id": artifact_id, "pred_json": json.dumps(pred, ensure_ascii=False)}


print('✅ ASSEMBLE defined')

## 10 · STEP 6 — DRIVE (run pipeline)
Same as original: load provenance → predict each id → write submission.csv

In [ ]:
def run(base=BASE, ids_csv=IDS, out=OUT, prov=PROV, limit=0, types=""):
    """
    Full pipeline run.
    limit : int  — process only first N ids (0 = all)
    types : str  — comma-separated type filter e.g. "WC,RC" ("" = all)
    """
    load_provenance(prov)
    only = {t.strip() for t in types.split(",") if t.strip()}
    ids  = [row[0] for row in csv.reader(open(ids_csv, newline=""))][1:]
    rows, n = [], 0

    for aid in ids:
        try:
            atype = route(aid)[0]
        except ValueError:
            rows.append({"artifact_id": aid, "pred_json": ""})
            continue

        if (only and atype not in only) or (limit and n >= limit):
            rows.append({"artifact_id": aid, "pred_json": ""})
            continue

        try:
            rows.append(assemble(aid, base))
        except Exception as e:
            print(f"[assemble-error] {aid}: {e}", file=sys.stderr)
            rows.append({"artifact_id": aid, "pred_json": ""})

        n += 1
        print(f"[{n}] {aid}", file=sys.stderr)

    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["artifact_id", "pred_json"])
        w.writeheader()
        w.writerows(rows)

    print(f"\n✅ Wrote {out}: {len(rows)} rows, {n} predicted")
    return rows


print('✅ DRIVE defined')

# Sample Test

In [ ]:
import os, json, glob

SAMPLE_IDS = [
    "BS-BBL-OPER-2567-01",
    "BN-MEGA1111-2567",
    "RC-TXN-202401-02004012",
]

for aid in SAMPLE_IDS:
    print("=" * 60)
    print(f"📄 artifact_id : {aid}")

    stype, subdir = route(aid)
    print(f"   type        : {stype}  (subdir: {subdir})")

    imgs = find_images(BASE, aid)
    if not imgs:
        print("   ⚠️  ไม่พบไฟล์รูป — ข้ามไป")
        continue
    for img in imgs:
        print(f"   📁 file       : {os.path.basename(img)}")

    # ── BS ต้องแยก header กับ txn ──────────────────────────────
    if stype == "BS":
        test_imgs = {
            "header": [im for im in imgs if "_header" in im],
            "txn":    [im for im in imgs if "_header" not in im],
        }
        for kind, kind_imgs in test_imgs.items():
            if not kind_imgs:
                continue
            img_path  = kind_imgs[0]
            type_key  = "BS_HEADER" if kind == "header" else "BS_TXN"
            rows_mode = kind == "txn"
            keys      = FIELDS[type_key]

            print(f"\n   [TyphoonOCR — {type_key}]")
            try:
                page_texts = typhoon_ocr_file(img_path)
                ocr_text   = "\n\n--- PAGE BREAK ---\n\n".join(page_texts).strip()
                preview    = ocr_text[:400].replace("\n", "\n      ")
                print(f"   ✅ OCR ({len(ocr_text)} chars):\n      {preview}")
                if len(ocr_text) > 400:
                    print(f"      ... (ตัดทอน {len(ocr_text)-400} ตัวอักษร)")
            except Exception as e:
                print(f"   ❌ OCR error: {e}")
                ocr_text = ""

            print(f"\n   [Qwen extract — {type_key}]")
            if not ocr_text:
                print("   ⚠️  ไม่มี OCR text — ข้ามไป")
                continue
            try:
                result = qwen_extract(ocr_text, type_key, keys, rows_mode)
                print("   ✅ JSON:\n      " +
                      json.dumps(result, ensure_ascii=False, indent=6)
                          .replace("\n", "\n      "))
            except Exception as e:
                print(f"   ❌ Qwen error: {e}")
        continue  # ข้าม logic ด้านล่างสำหรับ BS

    # ── Document types อื่นๆ ───────────────────────────────────
    img_path  = imgs[0]
    keys      = FIELDS[stype]
    rows_mode = False

    print("\n   [TyphoonOCR]")
    try:
        page_texts = typhoon_ocr_file(img_path)
        ocr_text   = "\n\n--- PAGE BREAK ---\n\n".join(page_texts).strip()
        preview    = ocr_text[:500].replace("\n", "\n      ")
        print(f"   ✅ OCR ({len(ocr_text)} chars):\n      {preview}")
        if len(ocr_text) > 500:
            print(f"      ... (ตัดทอน {len(ocr_text)-500} ตัวอักษร)")
    except Exception as e:
        print(f"   ❌ OCR error: {e}")
        ocr_text = ""

    print("\n   [Qwen extract]")
    if not ocr_text:
        print("   ⚠️  ไม่มี OCR text — ข้ามไป")
        continue
    try:
        result = qwen_extract(ocr_text, stype, keys, rows_mode)
        print("   ✅ JSON:\n      " +
              json.dumps(result, ensure_ascii=False, indent=6)
                  .replace("\n", "\n      "))
    except Exception as e:
        print(f"   ❌ Qwen error: {e}")

print("=" * 60)
print("✅ Sample test เสร็จสิ้น")

## 11 · Run

Choose one of the commands below:

In [ ]:
# ── Quick sanity test (5 items, WC only — no GPU memory pressure) ────────────
# run(limit=5, types="WC")

# ── Medium test (20 items, mixed types) ──────────────────────────────────────
# run(limit=20)

# ── Full run ─────────────────────────────────────────────────────────────────
run()

## 12 · Offline sanity checks (no model needed)
Verify ROUTE, PRE-FILL, and NORMALIZE logic without running OCR/LLM.

In [ ]:
# Routing & pre-fill
for aid in [
    "VI-V-013-INV-2567-226313",
    "WC-SKU-MASS-046-202405-10011040",
    "RC-TXN-202407-14003900",
]:
    print(aid, "->", route(aid), "| prefill", prefill_ids(aid))

print()
# Amount normalization
print("clean_amount tests:")
print(" ฿3,300.0    ->", clean_amount("฿3,300.0"))
print(" 12,720, 776.00 ->", clean_amount("12,720, 776.00"))
print(" ๑,๒๓๔.๕๖  ->", clean_amount("๑,๒๓๔.๕๖"))

In [ ]:
# Quick TyphoonOCR test on a single image (optional)
# Replace with an actual image path to verify OCR output

# test_img = "/content/drive/MyDrive/fahmai/.../some_receipt.png"
# text = typhoon_ocr_file(test_img)[0]
# print(text[:500])

In [ ]:
# Quick Qwen extraction test (optional)
# sample_text = "Receipt\nBranch: BKK-001\nDate: 15/07/2024\nTotal: 1,234.00 THB\nPayment: QR"
# result = qwen_extract(sample_text, "RC", FIELDS["RC"])
# print(json.dumps(result, ensure_ascii=False, indent=2))